# 08. Comparing algorithms

So far only LightGBM was tuned. Here we add three more and compare all four fairly:

| Model | Why include it | Needs |
|---|---|---|
| Logistic regression | Simple, interpretable linear benchmark | one-hot, log of skewed columns, imputation, scaling |
| Random forest | Different tree method (bagging, not boosting) | one-hot, imputation |
| XGBoost | Another boosting library, handles categories and NaN natively | nothing extra |
| LightGBM | Already tuned in notebook 06 | nothing extra |

**Fair comparison rules**
- Each model is tuned with Optuna (or a grid for logistic regression) on 3-fold CV, seed 42. The random forest is tuned on a 80,000-row subsample to keep it fast.
- Every model is scored at the end on the **same fresh 5-fold split (seed 7)**, so a model is never scored on the folds it was tuned on.

In [ ]:
import sys, json, time
sys.path.append("..")

import numpy as np
import pandas as pd
import optuna
import xgboost as xgb
import lightgbm as lgb
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

from src.features import clean, add_features, merge_rare, skewed_columns, log_transform, one_hot

optuna.logging.set_verbosity(optuna.logging.WARNING)

df = pd.read_csv("../data/raw/HC_application_train.csv")
y = df["TARGET"].reset_index(drop=True)
X = add_features(clean(df.drop(columns="TARGET")))
for name in ["bureau", "previous", "installments"]:
    X = X.join(pd.read_pickle(f"../data/processed/{name}.pkl"), on="SK_ID_CURR")
X = X.drop(columns="SK_ID_CURR").reset_index(drop=True)

# version for tree libraries with native categories
X_cat = X.copy()
for c in X_cat.select_dtypes(include=["object", "string"]):
    X_cat[c] = X_cat[c].astype("category")

# version for sklearn models: rare categories merged, one-hot, log of skewed columns
X_oh = one_hot(merge_rare(X))
X_oh = log_transform(X_oh, skewed_columns(X_oh)).replace([np.inf, -np.inf], np.nan).astype("float32")
print("native:", X_cat.shape, " one-hot:", X_oh.shape)

search_folds = list(StratifiedKFold(3, shuffle=True, random_state=42).split(X, y))
final_folds = list(StratifiedKFold(5, shuffle=True, random_state=7).split(X, y))
results = {}

## 1. Logistic regression: tune the regularisation strength `C`

In [ ]:
t0 = time.time()
lr_rows = []
for C in [0.005, 0.02, 0.05]:
    pipe = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=C, max_iter=150))
    lr_rows.append({"C": C, "AUC": cross_val_score(pipe, X_oh, y, cv=search_folds, scoring="roc_auc", n_jobs=3).mean()})
    print(lr_rows[-1], flush=True)
best_C = max(lr_rows, key=lambda r: r["AUC"])["C"]
print("best C:", best_C, f"({time.time()-t0:.0f}s)")

In [ ]:
def eval_sklearn(model, Xm):
    aucs = []
    for tr, va in final_folds:
        m = model.fit(Xm.iloc[tr], y.iloc[tr])
        aucs.append(roc_auc_score(y.iloc[va], m.predict_proba(Xm.iloc[va])[:, 1]))
    return np.array(aucs)

lr = make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), LogisticRegression(C=best_C, max_iter=500))
results["Logistic regression"] = eval_sklearn(lr, X_oh)
results["Logistic regression"].mean()

## 2. Random forest: tune on a subsample
Forests are slow on 300k rows x 650 columns, so the search uses 80,000 rows. The final scoring still uses all rows.

In [ ]:
t0 = time.time()
sub = np.random.RandomState(42).choice(len(X_oh), 80_000, replace=False)
Xs, ys = X_oh.iloc[sub].fillna(-999), y.iloc[sub]            # forests cannot take NaN; -999 sits outside every real range
sub_folds = list(StratifiedKFold(3, shuffle=True, random_state=42).split(Xs, ys))

def rf_objective(trial):
    m = RandomForestClassifier(
        n_estimators=150, n_jobs=-1, random_state=42,
        max_depth=trial.suggest_int("max_depth", 6, 20),
        min_samples_leaf=trial.suggest_int("min_samples_leaf", 5, 100, log=True),
        max_features=trial.suggest_float("max_features", 0.05, 0.4),
        class_weight=trial.suggest_categorical("class_weight", [None, "balanced_subsample"]),
    )
    return cross_val_score(m, Xs, ys, cv=sub_folds, scoring="roc_auc").mean()

rf_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
rf_study.optimize(rf_objective, n_trials=12)
print("best subsample AUC:", round(rf_study.best_value, 4), rf_study.best_params, f"({time.time()-t0:.0f}s)")

In [ ]:
t0 = time.time()
rf = RandomForestClassifier(n_estimators=400, n_jobs=-1, random_state=42, **rf_study.best_params)
X_rf = X_oh.fillna(-999)
results["Random forest"] = eval_sklearn(rf, X_rf)
print(results["Random forest"].mean(), f"({time.time()-t0:.0f}s)")

## 3. XGBoost: tune like LightGBM
Fast learning rate (0.1) with early stopping during the search, then 0.03 for the final scoring.

In [ ]:
t0 = time.time()
dtrain = xgb.DMatrix(X_cat, label=y, enable_categorical=True)

def xgb_objective(trial):
    p = {"objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "learning_rate": 0.1, "seed": 42,
         "max_depth": trial.suggest_int("max_depth", 3, 8),
         "min_child_weight": trial.suggest_int("min_child_weight", 1, 100, log=True),
         "colsample_bytree": trial.suggest_float("colsample_bytree", 0.1, 0.6),
         "subsample": trial.suggest_float("subsample", 0.6, 1.0),
         "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 10, log=True),
         "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 10, log=True),
         "max_cat_to_onehot": 1}
    res = xgb.cv(p, dtrain, num_boost_round=1000, folds=search_folds, early_stopping_rounds=50)
    return res["test-auc-mean"].iloc[-1]

xgb_study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42))
xgb_study.optimize(xgb_objective, n_trials=15)
print("best 3-fold AUC:", round(xgb_study.best_value, 4), xgb_study.best_params, f"({time.time()-t0:.0f}s)")

In [ ]:
t0 = time.time()
xp = {"objective": "binary:logistic", "eval_metric": "auc", "tree_method": "hist", "learning_rate": 0.03, "seed": 42,
      "max_cat_to_onehot": 1, **xgb_study.best_params}
aucs, rounds = [], []
for tr, va in final_folds:
    dt = xgb.DMatrix(X_cat.iloc[tr], y.iloc[tr], enable_categorical=True)
    dv = xgb.DMatrix(X_cat.iloc[va], y.iloc[va], enable_categorical=True)
    m = xgb.train(xp, dt, num_boost_round=3000, evals=[(dv, "va")], early_stopping_rounds=100, verbose_eval=False)
    aucs.append(m.best_score); rounds.append(m.best_iteration + 1)
results["XGBoost"] = np.array(aucs)
print(results["XGBoost"].mean(), "trees:", int(np.mean(rounds)), f"({time.time()-t0:.0f}s)")

## 4. LightGBM: re-scored with the tuned parameters from notebook 06 on the same folds

In [ ]:
lp = {k: v for k, v in json.load(open("../models/best_params.json")).items() if k != "n_estimators"}
lp.update({"objective": "binary", "metric": "auc", "verbosity": -1, "seed": 42})
aucs = []
for tr, va in final_folds:
    dt = lgb.Dataset(X_cat.iloc[tr], y.iloc[tr]); dv = lgb.Dataset(X_cat.iloc[va], y.iloc[va], reference=dt)
    m = lgb.train(lp, dt, num_boost_round=3000, valid_sets=[dv], callbacks=[lgb.early_stopping(100, verbose=False)])
    aucs.append(roc_auc_score(y.iloc[va], m.predict(X_cat.iloc[va], num_iteration=m.best_iteration)))
results["LightGBM"] = np.array(aucs)
results["LightGBM"].mean()

## Final comparison (same 5 folds, seed 7)

In [ ]:
summary = pd.DataFrame({"AUC mean": {k: v.mean() for k, v in results.items()},
                        "AUC std": {k: v.std() for k, v in results.items()}}).sort_values("AUC mean", ascending=False)
summary["vs best"] = summary["AUC mean"] - summary["AUC mean"].max()
summary.round(4)

In [ ]:
import matplotlib.pyplot as plt
order = summary.index
plt.figure(figsize=(7, 3.5))
plt.boxplot([results[k] for k in order], tick_labels=order)
plt.ylabel("ROC-AUC per fold"); plt.title("Five folds per model, same splits")
plt.tight_layout(); plt.savefig("../assets/model_comparison.png", dpi=150); plt.show()

In [ ]:
json.dump({"logistic_C": best_C, "random_forest": rf_study.best_params, "xgboost": xgb_study.best_params,
           "final_auc": summary["AUC mean"].round(4).to_dict()}, open("../models/model_comparison.json", "w"), indent=2)